# Страж — EDA & Feature Engineering Plan
**Цель:** побить PR-AUC 0.1706

## Структура проекта
```
hak/
├── main_data/          ← sample_submit.csv, train_labels.parquet
├── Pre-train/          ← история операций (2023-10-01 → 2024-09-30), без меток
├── Pre-test/           ← операции (2025-06-01 → 2025-08-09), без меток
├── train/              ← операции (2024-10-01 → 2025-05-31) + метки
├── test/               ← финальный день для каждого клиента
├── 01_eda.ipynb        ← этот файл
├── 02_features.ipynb   ← feature engineering
├── 03_train.ipynb      ← обучение моделей
└── 04_submit.ipynb     ← финальный сабмит
```

## Общий план атаки

### Этап 1 — EDA (этот ноутбук)
- Размеры датасетов, типы колонок
- Дисбаланс классов
- Распределения ключевых фич
- Анализ target=1 vs target=0 vs без метки

### Этап 2 — Feature Engineering (02_features.ipynb)
- Velocity features (временные окна)
- Device fingerprinting
- Поведенческие аномалии
- Сохранение фич в parquet

### Этап 3 — Обучение (03_train.ipynb)
- LightGBM GPU baseline
- CatBoost GPU
- Ансамбль / стекинг
- TimeSeriesSplit CV

### Этап 4 — Сабмит (04_submit.ipynb)
- Применение модели к test
- Формирование CSV

In [ ]:
import polars as pl
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Пути
ROOT = Path('/home/vadim/PyPr/hak')
DATA = ROOT / 'main_data'
PRETRAIN = ROOT / 'Pre-train'
PRETEST = ROOT / 'Pre-test'
# TRAIN = ROOT / 'train'  # раскомментируй когда появится папка
# TEST = ROOT / 'test'

print('Всё импортировано')

## 1. Смотрим что скачалось

In [ ]:
import os
for folder in [DATA, PRETRAIN, PRETEST]:
    print(f'\n--- {folder.name} ---')
    if folder.exists():
        for f in sorted(folder.iterdir()):
            size_mb = f.stat().st_size / 1024**2
            print(f'  {f.name:40s} {size_mb:8.1f} MB')
    else:
        print('  [папка не существует]')

## 2. Train labels

In [ ]:
labels = pl.read_parquet(DATA / 'train_labels.parquet')
print('Shape:', labels.shape)
print(labels.head(10))
print('\nTarget distribution:')
print(labels['target'].value_counts().sort('target'))

## 3. Читаем кусок Pre-train (когда файлы появятся)

In [ ]:
# Раскомментировать после загрузки файлов
# pretrain_files = sorted(PRETRAIN.glob('*.parquet'))
# print('Pre-train files:', pretrain_files)
# 
# # Читаем первый файл для EDA
# df = pl.read_parquet(pretrain_files[0])
# print('Shape:', df.shape)
# print('Columns:', df.columns)
# print(df.head(5))

## 4. Schema и типы данных

In [ ]:
# После загрузки данных:
# print(df.dtypes)
# print('\nNull counts:')
# print(df.null_count())
# print('\nMemory usage:', df.estimated_size('mb'), 'MB')

## 5. Анализ ключевых признаков
### Security flags — главные сигналы фрода

In [ ]:
# SECURITY_FLAGS = ['compromised', 'web_rdp_connection', 'phone_voip_call_state', 'developer_tools']
# 
# fig, axes = plt.subplots(1, len(SECURITY_FLAGS), figsize=(16, 4))
# for ax, col in zip(axes, SECURITY_FLAGS):
#     # Сравниваем распределение для fraud vs normal
#     ...
# plt.tight_layout()
# plt.show()

## 6. Временные паттерны

In [ ]:
# df_labeled = df.join(labels, on=['event_id'], how='left')
# 
# # Hour of day
# df_labeled = df_labeled.with_columns(
#     pl.col('event_dttm').dt.hour().alias('hour'),
#     pl.col('event_dttm').dt.weekday().alias('weekday'),
# )
# 
# # Фрод чаще ночью?
# fraud_by_hour = df_labeled.filter(pl.col('target') == 1).group_by('hour').agg(pl.len())
# normal_by_hour = df_labeled.filter(pl.col('target').is_null()).group_by('hour').agg(pl.len())